<a href="https://colab.research.google.com/github/brucenguyen0302-code/warehouse-pick-optimiser/blob/main/notebooks/Nguyen_LeBinh_AT1_1_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AT1 Example I - Warehouse Pick-Path Optimisation

**Le Binh Nguyen (26205599)** - 42172 Introduction to AI

This notebook solves a route optimisation problem in a real warehouse. A picker must
collect every item in a picking wave from shelves spread across the warehouse floor and
return to the packing station. The goal is to find the order of stops that gives the
shortest total walking distance.

Two AI techniques are compared: **Simulated Annealing** and **Genetic Algorithm**. A\* search is used to measure true walking distance around the racking.

**Data:** de Assis, R. F. et al. *Order Picking Dataset from a Warehouse of a Footwear
Manufacturing Company.* Mendeley Data, V1. https://doi.org/10.17632/pf2w725pw3.1

## 1. Setup and data

### 1.1 Connect to the data

This cell
mounts Drive, then searches it for `Storage_Location.csv` and uses whichever folder
contains that file as the data directory. Searching instead of hardcoding a path means the
notebook still works if the folder is moved or renamed. If the file is not found it raises
an error immediately, so a missing file is caught here rather than causing a confusing
failure several cells later.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, glob

NEEDED = ['Storage_Location.csv', 'Support_Points_Navigation.csv',
          'Picking_Wave.csv', 'Customer_Order.csv', 'Product.csv']

hits = glob.glob('/content/gdrive/MyDrive/**/Storage_Location.csv', recursive=True)
if not hits:
    raise FileNotFoundError(
        'Storage_Location.csv not found anywhere in your Drive. '
        'Upload the five CSVs to a folder in Google Drive.')

DATA_DIR = os.path.dirname(hits[0])
print(f'DATA_DIR = {DATA_DIR}\n')

for name in NEEDED:
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        print(f'  found  {os.path.getsize(path)/1e6:7.2f} MB  {name}')
    else:
        print(f'  MISSING              {name}')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
DATA_DIR = /content/gdrive/MyDrive/IntroToAI/AT1_I/data

  found     0.07 MB  Storage_Location.csv
  found     0.00 MB  Support_Points_Navigation.csv
  found     9.68 MB  Picking_Wave.csv
  found     9.46 MB  Customer_Order.csv
  found     0.00 MB  Product.csv


### 1.2 Load the warehouse layout

Two files describe the building. `Storage_Location.csv` gives the coordinates of all 2,292
storage locations. `Support_Points_Navigation.csv` gives the navigation waypoints the
picker travels along.

The navigation file uses carriage-return line endings rather than newlines, so it appears
as one continuous line in most tools. A regular expression is used to pull out each
coordinate and its label, which works regardless of how the lines are terminated. Both
files begin with an invisible byte-order mark, which is why `encoding='utf-8-sig'` is
needed - without it the first column name is corrupted.

The output shows the structure of the building: 17 horizontal aisles and 3 vertical
corridors.

In [2]:
import re
import numpy as np
import pandas as pd

storage = pd.read_csv(os.path.join(DATA_DIR, 'Storage_Location.csv'))
storage['location'] = storage['originalLocation'].astype(str).str.strip()

nav_text = open(os.path.join(DATA_DIR, 'Support_Points_Navigation.csv'),
                encoding='utf-8-sig').read()
nav = pd.DataFrame(
    re.findall(r'\(([-\d.]+),\s*([-\d.]+),\s*([-\d.]+)\);([A-Z]+)-(\d+)', nav_text),
    columns=['x', 'y', 'z', 'rail', 'index']).astype({'x': float, 'y': float})

AISLE_Y    = np.array(sorted(nav['y'].unique()))
CORRIDOR_X = np.array(sorted(nav['x'].unique()))

print(f'storage locations   : {len(storage):,}')
print(f'aisles (horizontal) : {len(AISLE_Y)}  at y = {AISLE_Y.astype(int).tolist()}')
print(f'corridors (vertical): {len(CORRIDOR_X)}  at x = {CORRIDOR_X.astype(int).tolist()}')

storage locations   : 2,292
aisles (horizontal) : 17  at y = [-29, 61, 151, 241, 331, 421, 511, 631, 751, 841, 931, 1021, 1111, 1201, 1291, 1381, 1471]
corridors (vertical): 3  at x = [66, 403, 686]


### 1.3 Convert storage locations into walking positions

Stock sits inside the racking, but the picker stands in the aisle next to it. Each storage
location is therefore mapped to the position the picker actually walks to: the same x
coordinate, at the nearest aisle.

This matters because many locations share one standing position. The four rack levels at a
single spot, and the rows on either side of an aisle, all collapse to the same stop. The
2,292 storage locations reduce to 466 distinct floor positions, so a wave listing 20 items
may require only 14 places to walk to.

**Assumption:** vertical level (z) is ignored and the problem is treated in two dimensions,
because reaching a higher shelf affects picking time but not walking distance.

In [3]:
nearest = np.abs(storage['y'].values[:, None] - AISLE_Y).argmin(axis=1)
storage['face_x'] = storage['x']
storage['face_y'] = AISLE_Y[nearest]
storage['face']   = list(zip(storage['face_x'], storage['face_y']))

print(f'{len(storage):,} storage locations collapse to '
      f'{storage["face"].nunique():,} distinct floor stops')
print('reach-in distance from aisle to rack:',
      sorted({int(d) for d in abs(storage['y'] - storage['face_y'])}), 'units')

2,292 storage locations collapse to 466 distinct floor stops
reach-in distance from aisle to rack: [27, 29, 31, 57, 59] units


### 1.4 Load and clean the picking waves

This loads all 215,192 pick lines and attaches a floor position to each one.

11% of pick lines refer to locations that do not appear in the coordinate file. These are
only 22 distinct codes, almost all `RC-01`, and they belong to a zone of the warehouse for
which the dataset authors did not publish coordinates. Those lines are removed because
their position is unknown and they cannot be routed.

The cell prints how many lines are dropped and which codes they are, so the exclusion is
visible and can be justified rather than happening silently.

In [4]:
waves_raw = pd.read_csv(os.path.join(DATA_DIR, 'Picking_Wave.csv'),
                        sep=';', encoding='utf-8-sig')
waves_raw.columns = [c.strip() for c in waves_raw.columns]
waves_raw['location'] = waves_raw['locations'].astype(str).str.strip()

face_of = dict(zip(storage['location'], storage['face']))
waves_raw['face'] = waves_raw['location'].map(face_of)

unmapped = waves_raw['face'].isna()
print(f'pick lines total   : {len(waves_raw):,}')
print(f'unmapped locations : {unmapped.sum():,} ({unmapped.mean():.1%}) '
      f'across {waves_raw.loc[unmapped, "location"].nunique()} distinct codes')
print('  most common      :',
      waves_raw.loc[unmapped, 'location'].value_counts().head(3).to_dict())

waves_clean = waves_raw[~unmapped].copy()

pick lines total   : 215,192
unmapped locations : 23,609 (11.0%) across 22 distinct codes
  most common      : {'RC-01': 22979, 'RC-04': 361, 'RC-05': 100}


### 1.5 Build routable waves and split by date

Individual customer orders in this dataset average fewer than two distinct products, so
there is no meaningful route to optimise at order level. This is exactly why the company
groups orders into picking waves before sending a picker out, and the wave is therefore the
unit being optimised.

Waves with fewer than 8 stops are excluded, because with so few stops the problem can be
solved by inspection and does not need a metaheuristic.

Each wave is dated from its earliest customer order, and the waves are split
chronologically at 1 July 2023. Earlier waves are used for tuning parameters and later
waves are held back entirely. This gives a genuine test on future data, rather than a
random split which would let information from the test period leak into tuning.

In [5]:
orders = pd.read_csv(os.path.join(DATA_DIR, 'Customer_Order.csv'),
                     sep=';', encoding='utf-8-sig')
orders.columns = [c.strip() for c in orders.columns]
orders['date'] = pd.to_datetime(orders['creationDate'],
                                format='%d/%m/%Y %H:%M', errors='coerce')
wave_date = orders.groupby('waveNumber')['date'].min()

MIN_STOPS  = 8
SPLIT_DATE = pd.Timestamp('2023-07-01')

stops = waves_clean.groupby('waveNumber')['face'].apply(lambda s: list(dict.fromkeys(s)))
waves = pd.DataFrame({'stops': stops, 'n_stops': stops.apply(len)})
waves['date']  = wave_date.reindex(waves.index)
waves = waves[(waves['n_stops'] >= MIN_STOPS) & waves['date'].notna()]
waves['split'] = np.where(waves['date'] < SPLIT_DATE, 'tune', 'test')

print(f'routable waves (>= {MIN_STOPS} stops): {len(waves):,}')
print(f'  stops per wave : mean {waves.n_stops.mean():.1f}  '
      f'median {waves.n_stops.median():.0f}  max {waves.n_stops.max()}')
print(f'  date range     : {waves.date.min().date()} to {waves.date.max().date()}')
print(waves['split'].value_counts().to_string())

routable waves (>= 8 stops): 5,858
  stops per wave : mean 13.8  median 14  max 24
  date range     : 2023-01-05 to 2023-10-19
split
tune    3633
test    2225


## 2. Navigation: measuring real walking distance

### 2.1 Build the aisle network

The picker cannot walk in straight lines because racking is in the way. Movement is
restricted to 17 horizontal aisles and 3 vertical corridors, and the corridors are the only
places where the picker can change aisle.

This cell builds that movement network as a weighted graph. Nodes are positions where the
picker can stand, and edges are the aisle and corridor segments connecting them, weighted
by their length. The packing station is placed at corridor point CC-08, which is where the
dataset authors mark it on their layout drawing.

In [6]:
import heapq, itertools, random

faces = sorted(set(storage['face']))
rail_levels = {x: sorted(nav.loc[nav['x'] == x, 'y'].unique()) for x in CORRIDOR_X}

nodes_on_aisle = {y: set() for y in AISLE_Y}
for x, y in faces:
    nodes_on_aisle[y].add(x)
for x, ys in rail_levels.items():
    for y in ys:
        nodes_on_aisle[y].add(x)

graph = {}
def connect(a, b, weight):
    graph.setdefault(a, []).append((b, weight))
    graph.setdefault(b, []).append((a, weight))

for y, xs in nodes_on_aisle.items():              # walk along an aisle
    xs = sorted(xs)
    for x1, x2 in zip(xs, xs[1:]):
        connect((x1, y), (x2, y), x2 - x1)

for x, ys in rail_levels.items():                 # change aisle at a corridor
    for y1, y2 in zip(ys, ys[1:]):
        connect((x, y1), (x, y2), y2 - y1)

DEPOT = (403.0, 631.0)   # corridor point CC-08, per the dataset's layout figure

print(f'network : {len(graph):,} nodes, {sum(len(v) for v in graph.values())//2:,} edges')
print(f'depot   : {DEPOT}  present in network: {DEPOT in graph}')

network : 500 nodes, 524 edges
depot   : (403.0, 631.0)  present in network: True


### 2.2 A\* search

A\* finds the shortest walking distance between two positions in the network. It uses
Manhattan distance as its heuristic, which is admissible here because all movement is along
horizontal or vertical segments, so the straight-line estimate can never exceed the true
distance. Admissibility is what guarantees A\* returns an optimal path.

Dijkstra's algorithm is also implemented. It is not a competing technique - it is an
independently written reference used in the next cell to check that A\* is correct.

In [7]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(start, goal):
    """Shortest walking distance between two positions.

    Returns (distance, number of nodes expanded).
    """
    if start == goal:
        return 0.0, 0

    counter  = itertools.count()
    frontier = [(manhattan(start, goal), 0.0, next(counter), start)]
    best_g   = {start: 0.0}
    closed   = set()

    while frontier:
        _, g, _, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        if node == goal:
            return g, len(closed)
        for nxt, weight in graph[node]:
            tentative = g + weight
            if tentative < best_g.get(nxt, float('inf')):
                best_g[nxt] = tentative
                heapq.heappush(frontier, (tentative + manhattan(nxt, goal),
                                          tentative, next(counter), nxt))
    return float('inf'), len(closed)


def dijkstra_from(start):
    """Exact distance from start to every node. Reference for checking A*."""
    dist, frontier, closed = {start: 0.0}, [(0.0, start)], set()
    while frontier:
        d, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        for nxt, weight in graph[node]:
            if d + weight < dist.get(nxt, float('inf')):
                dist[nxt] = d + weight
                heapq.heappush(frontier, (dist[nxt], nxt))
    return dist

### 2.3 Verify A\*, then precompute all distances

A\* is checked against Dijkstra on 300 random pairs of positions. The two must agree
exactly, and the heuristic is confirmed admissible on every pair. Because the two
algorithms were written separately, a bug would have to appear in both to go unnoticed.

A distance matrix is then built once for all 466 positions, so the optimisation stage can
look distances up instead of searching repeatedly. A single simulated annealing run needs
hundreds of thousands of distance lookups; re-running A\* for each one would take minutes
instead of milliseconds.

The matrix is checked for symmetry, a zero diagonal and finite values. An earlier version
of this code passed visual inspection while being silently wrong in one row, which is why
these assertions are here.

In [8]:
rng   = random.Random(0)
pairs = [(rng.choice(faces), rng.choice(faces)) for _ in range(300)]

fields, mismatch, expanded = {}, 0, []
for s, g in pairs:
    if s not in fields:
        fields[s] = dijkstra_from(s)
    cost, n = astar(s, g)
    expanded.append(n)
    if abs(cost - fields[s][g]) > 1e-9:
        mismatch += 1

print(f'A* vs Dijkstra on 300 pairs : {300 - mismatch} identical, {mismatch} mismatched')
print(f'heuristic admissible        : '
      f'{all(manhattan(s, g) <= astar(s, g)[0] + 1e-9 for s, g in pairs)}')
print(f'A* nodes expanded           : mean {np.mean(expanded):.0f} of {len(graph)}')
ratio = [astar(s, g)[0] / max(manhattan(s, g), 1) for s, g in pairs if s != g]
print(f'walking vs straight line    : mean {np.mean(ratio):.2f}x, max {np.max(ratio):.2f}x')

POINTS = [DEPOT] + [f for f in faces if f != DEPOT]
INDEX  = {p: i for i, p in enumerate(POINTS)}
assert len(INDEX) == len(POINTS), 'duplicate point in POINTS'

DIST = np.zeros((len(POINTS), len(POINTS)))
for p in POINTS:
    field = dijkstra_from(p)
    DIST[INDEX[p]] = [field.get(q, np.inf) for q in POINTS]

print(f'\ndistance matrix {DIST.shape[0]}x{DIST.shape[1]}')
print(f'  symmetric     : {np.allclose(DIST, DIST.T)}')
print(f'  zero diagonal : {np.allclose(np.diag(DIST), 0)}')
print(f'  all finite    : {np.isfinite(DIST).all()}')
print(f'  longest walk  : {DIST.max():.0f} units = {DIST.max()/10:.0f} m')

A* vs Dijkstra on 300 pairs : 300 identical, 0 mismatched
heuristic admissible        : True
A* nodes expanded           : mean 54 of 500
walking vs straight line    : mean 1.17x, max 3.55x

distance matrix 466x466
  symmetric     : True
  zero diagonal : True
  all finite    : True
  longest walk  : 2080 units = 208 m


## 3. Objective function and baselines

### 3.1 Defining the objective

A route is a sequence of stops. Its cost is the total walking distance from the packing
station, through every stop in that order, and back to the station. Distances come from the
matrix built in Section 2, so evaluating a route is a handful of array lookups.

The objective is wrapped in a class that counts how many times it is called. Simulated
annealing and the genetic algorithm evaluate different numbers of routes per iteration, so
comparing them fairly requires giving them the same number of objective evaluations rather
than the same number of iterations. The counter is what makes that possible.

In [10]:
def route_length(order):
    """Total walking distance: depot -> stops in the given order -> depot."""
    total = DIST[0, order[0]]
    for a, b in zip(order, order[1:]):
        total += DIST[a, b]
    return total + DIST[order[-1], 0]


def is_valid_route(order, stops):
    """A route must visit every stop exactly once and nothing else."""
    return len(order) == len(stops) and set(order) == set(stops)


class CountedObjective:
    """Wraps the objective and counts calls, so SA and GA can be given equal budgets."""
    def __init__(self, fn):
        self.fn, self.calls = fn, 0
    def __call__(self, order):
        self.calls += 1
        return self.fn(order)
    def reset(self):
        self.calls = 0


waves['route'] = waves.stops.apply(lambda s: [INDEX[p] for p in s])

example = waves.iloc[0]
print(f'example wave {example.name}: {example.n_stops} stops')
print(f'  listed order length : {route_length(example.route):,.0f} units')
print(f'  valid route         : {is_valid_route(example.route, example.route)}')

example wave 33168: 21 stops
  listed order length : 6,872 units
  valid route         : True


### 3.2 Three reference points

Three baselines are needed before any result can be interpreted.

**Random order** shows what happens with no organisation at all. It is a weak baseline and
beating it proves very little, but it is included because several published studies use it,
and comparing it with the next baseline shows how much that choice inflates results.

**Listed order** is the sequence the picks appear in within the dataset. Checking 300 waves
confirms this is not simply sorted by product or by location, so it reflects the order the
company's own system produced. This is the honest baseline: it represents what is being
done today, and it is what any improvement must be measured against.

**Nearest neighbour** repeatedly walks to the closest unvisited stop. It is a greedy
heuristic, not a metaheuristic, and it shows how much of the available improvement can be
captured by a simple rule.

In [11]:
def random_order(stops, seed):
    order = list(stops)
    random.Random(seed).shuffle(order)
    return order


def nearest_neighbour(stops):
    """Repeatedly walk to whichever unvisited stop is closest."""
    unvisited, current, route = set(stops), 0, []
    while unvisited:
        nxt = min(unvisited, key=lambda j: DIST[current, j])
        route.append(nxt)
        unvisited.discard(nxt)
        current = nxt
    return route


tune = waves[waves.split == 'tune']

baselines = pd.DataFrame(index=tune.index)
baselines['listed'] = tune.route.apply(route_length)
baselines['random'] = tune.route.apply(
    lambda r: np.mean([route_length(random_order(r, s)) for s in range(5)]))
baselines['nearest_neighbour'] = tune.route.apply(
    lambda r: route_length(nearest_neighbour(r)))

print(f'mean route length over {len(tune):,} tuning waves')
for c in ['random', 'listed', 'nearest_neighbour']:
    print(f'  {c:18s}: {baselines[c].mean():8,.0f} units')

mean route length over 3,633 tuning waves
  random            :    6,796 units
  listed            :    3,787 units
  nearest_neighbour :    3,318 units


### 3.3 The true optimum

For waves with few enough stops the shortest possible route can be computed exactly using
the Held-Karp dynamic programming algorithm. Its cost grows as O(n^2 * 2^n), so it is only
practical up to about 12 stops, but that is enough to establish ground truth on part of the
data.

This is what makes the results interpretable. Without it, we could only say one method beats
another. With it, we can say how far each method is from the best route that exists.

In [12]:
def held_karp(stops):
    """Exact shortest route by dynamic programming. Feasible up to about 12 stops."""
    n = len(stops)
    dp = {(1 << i, i): (DIST[0, stops[i]], -1) for i in range(n)}
    for size in range(2, n + 1):
        for subset in itertools.combinations(range(n), size):
            mask = sum(1 << i for i in subset)
            for last in subset:
                prev = mask ^ (1 << last)
                dp[(mask, last)] = min(
                    (dp[(prev, k)][0] + DIST[stops[k], stops[last]], k)
                    for k in subset if k != last)
    full = (1 << n) - 1
    cost, last = min((dp[(full, i)][0] + DIST[stops[i], 0], i) for i in range(n))
    order, mask = [], full
    while last != -1:
        order.append(stops[last])
        nxt = dp[(mask, last)][1]
        mask ^= 1 << last
        last = nxt
    return order[::-1], cost


EXACT_MAX_STOPS, EXACT_SAMPLE = 11, 150
small = tune[tune.n_stops <= EXACT_MAX_STOPS].head(EXACT_SAMPLE)

optimum = small.route.apply(lambda r: held_karp(r)[1])

print(f'exact optimum on {len(small)} waves of <= {EXACT_MAX_STOPS} stops')
print('gap above the true optimum:')
for c in ['random', 'listed', 'nearest_neighbour']:
    print(f'  {c:18s}: {(baselines.loc[small.index, c] / optimum - 1).mean():6.1%}')

exact optimum on 150 waves of <= 11 stops
gap above the true optimum:
  random            :  76.0%
  listed            :  17.1%
  nearest_neighbour :   4.6%


### 3.4 Hill climbing

Hill climbing is the third local search algorithm from Lab 3. It starts from a random route,
repeatedly swaps two stops, and keeps a swap only when it shortens the route. Because it
never accepts a worse route it becomes trapped in local optima, so it is restarted from
several random starting points and the best result kept.

It is included as a reference point rather than as one of the compared techniques. It shows
what plain local search achieves, which is the bar simulated annealing must clear to justify
its extra machinery.

In [13]:
def hill_climbing(stops, objective, iterations=1500, restarts=20, seed=0):
    """Lab 3 hill climbing: swap two stops, keep the change only if it improves."""
    rng = random.Random(seed)
    best_route, best_cost = None, float('inf')
    for _ in range(restarts):
        current = list(stops)
        rng.shuffle(current)
        current_cost = objective(current)
        for _ in range(iterations):
            i, j = rng.sample(range(len(current)), 2)
            neighbour = current[:]
            neighbour[i], neighbour[j] = neighbour[j], neighbour[i]
            neighbour_cost = objective(neighbour)
            if neighbour_cost < current_cost:
                current, current_cost = neighbour, neighbour_cost
        if current_cost < best_cost:
            best_route, best_cost = current, current_cost
    return best_route, best_cost


HC_SAMPLE = 100
hc_waves = tune.head(HC_SAMPLE)

hc_costs, hc_calls = [], []
for r in hc_waves.route:
    obj = CountedObjective(route_length)
    _, cost = hill_climbing(r, obj, seed=1)
    hc_costs.append(cost)
    hc_calls.append(obj.calls)

hill = pd.Series(hc_costs, index=hc_waves.index)
print(f'hill climbing on {HC_SAMPLE} waves, '
      f'{np.mean(hc_calls):,.0f} objective evaluations each')

hill climbing on 100 waves, 30,020 objective evaluations each


### 3.5 Baseline summary

The table compares all four reference points on the same 100 waves, measured against the
company's existing order.

The important comparison is between the random and listed baselines. The company's own
picking order is already far better than random, which means any study reporting improvement
against a random baseline is measuring mostly the absence of organisation rather than the
value of its algorithm. Measured honestly, against what the warehouse actually does today,
the available improvement is much smaller and much harder to win.

In [14]:
idx = hc_waves.index
ref = baselines.loc[idx, 'listed'].mean()

summary = pd.DataFrame({
    'mean route': [baselines.loc[idx, 'random'].mean(),
                   baselines.loc[idx, 'listed'].mean(),
                   baselines.loc[idx, 'nearest_neighbour'].mean(),
                   hill.mean()],
}, index=['random order', 'listed order (company)', 'nearest neighbour', 'hill climbing'])
summary['vs listed order'] = (summary['mean route'] / ref - 1).map('{:+.1%}'.format)
summary['mean route'] = summary['mean route'].map('{:,.0f}'.format)
print(summary.to_string())

                       mean route vs listed order
random order                7,058          +72.3%
listed order (company)      4,097           +0.0%
nearest neighbour           3,434          -16.2%
hill climbing               3,306          -19.3%
